# 04 — Hyperparameter Tuning & Model Explainability

**Step 1:** Tune the best model from Notebook 03 using Optuna (Bayesian optimisation).
**Step 2:** Explain predictions using SHAP values.

Optuna uses the **validation set** for scoring. Final evaluation is on the **test set** (never seen during tuning).

In [ ]:
import numpy as np
import pandas as pd
import json, joblib, os
import matplotlib.pyplot as plt
import seaborn as sns
import optuna
import shap
from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score,
                              recall_score, precision_score, classification_report)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import warnings
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110
SEED = 42
SCALE_POS_WEIGHT = 900773 / 99227

In [ ]:
X_train = np.load('artifacts/X_train.npy')
X_val   = np.load('artifacts/X_val.npy')
X_test  = np.load('artifacts/X_test.npy')
y_train = np.load('artifacts/y_train.npy')
y_val   = np.load('artifacts/y_val.npy')
y_test  = np.load('artifacts/y_test.npy')

with open('artifacts/proc_cols.json') as f:
    PROC_COLS = json.load(f)

compare_df = pd.read_csv('artifacts/model_comparison.csv', index_col='model')
best_model_name = compare_df['roc_auc'].idxmax()
print(f'Tuning: {best_model_name}')
print(compare_df[['roc_auc','pr_auc','f1_churn']].to_string())

## 1. Optuna Hyperparameter Tuning

Bayesian optimisation over 60 trials. Objective: maximise **ROC-AUC on the validation set**.

Searching XGBoost and LightGBM parameter spaces — whichever was the winner.

In [ ]:
# XGBoost objective
def objective_xgb(trial):
    params = {
        'n_estimators':      trial.suggest_int('n_estimators', 200, 800),
        'max_depth':         trial.suggest_int('max_depth', 3, 9),
        'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'subsample':         trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight':  trial.suggest_int('min_child_weight', 10, 100),
        'gamma':             trial.suggest_float('gamma', 0, 5),
        'reg_alpha':         trial.suggest_float('reg_alpha', 0, 2),
        'reg_lambda':        trial.suggest_float('reg_lambda', 0.5, 5),
        'scale_pos_weight':  SCALE_POS_WEIGHT,
        'random_state': SEED, 'n_jobs': -1, 'verbosity': 0,
        'eval_metric': 'aucpr', 'early_stopping_rounds': 15,
    }
    m = XGBClassifier(**params)
    m.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    return roc_auc_score(y_val, m.predict_proba(X_val)[:, 1])

In [ ]:
# LightGBM objective
def objective_lgbm(trial):
    params = {
        'n_estimators':      trial.suggest_int('n_estimators', 200, 800),
        'num_leaves':        trial.suggest_int('num_leaves', 20, 150),
        'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'subsample':         trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_samples': trial.suggest_int('min_child_samples', 50, 300),
        'reg_alpha':         trial.suggest_float('reg_alpha', 0, 2),
        'reg_lambda':        trial.suggest_float('reg_lambda', 0.5, 5),
        'is_unbalance': True,
        'random_state': SEED, 'n_jobs': -1, 'verbosity': -1,
    }
    m = LGBMClassifier(**params)
    m.fit(X_train, y_train, eval_set=[(X_val, y_val)])
    return roc_auc_score(y_val, m.predict_proba(X_val)[:, 1])

In [ ]:
N_TRIALS = 60
print(f'Running {N_TRIALS} Optuna trials for {best_model_name}...')

objective = objective_xgb if 'XGB' in best_model_name else objective_lgbm
study = optuna.create_study(direction='maximize',
                             sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

print(f'\nBest trial:  ROC-AUC = {study.best_value:.5f}')
print(f'Best params:')
for k, v in study.best_params.items():
    print(f'  {k}: {v}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Optimisation history
trials_df = study.trials_dataframe()
axes[0].plot(trials_df['number'], trials_df['value'], alpha=0.5, color='steelblue', linewidth=1)
axes[0].plot(trials_df['number'],
             trials_df['value'].cummax(), color='tomato', linewidth=2, label='Best so far')
axes[0].set_title('Optuna Optimisation History', fontweight='bold')
axes[0].set_xlabel('Trial'); axes[0].set_ylabel('Val ROC-AUC')
axes[0].legend()

# Parameter importance (top 8)
try:
    importances = optuna.importance.get_param_importances(study)
    imp_s = pd.Series(importances).sort_values(ascending=True).tail(8)
    imp_s.plot(kind='barh', ax=axes[1], color='darkorange', edgecolor='white', alpha=0.85)
    axes[1].set_title('Optuna — Hyperparameter Importance', fontweight='bold')
except Exception:
    axes[1].set_title('Param importance unavailable')

plt.tight_layout(); plt.show()

## 2. Retrain Best Model with Tuned Parameters

In [ ]:
best_params = study.best_params.copy()
best_params.update({'random_state': SEED, 'n_jobs': -1})

if 'XGB' in best_model_name:
    best_params.update({'scale_pos_weight': SCALE_POS_WEIGHT,
                        'verbosity': 0, 'eval_metric': 'aucpr',
                        'early_stopping_rounds': 20})
    tuned_model = XGBClassifier(**best_params)
    tuned_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
else:
    best_params.update({'is_unbalance': True, 'verbosity': -1})
    tuned_model = LGBMClassifier(**best_params)
    tuned_model.fit(X_train, y_train, eval_set=[(X_val, y_val)])

print('Tuned model trained.')

In [ ]:
# Tune threshold on validation set
prob_val  = tuned_model.predict_proba(X_val)[:, 1]
prob_test = tuned_model.predict_proba(X_test)[:, 1]

thresholds = np.linspace(0.1, 0.9, 81)
f1s = [f1_score(y_val, (prob_val >= t).astype(int), pos_label=1, zero_division=0)
       for t in thresholds]
best_t = thresholds[np.argmax(f1s)]
y_pred = (prob_test >= best_t).astype(int)

print(f'Optimal threshold: {best_t:.3f}')
print()
print('=== FINAL TEST SET RESULTS (tuned model) ===')
print(f'ROC-AUC  : {roc_auc_score(y_test, prob_test):.4f}')
print(f'PR-AUC   : {average_precision_score(y_test, prob_test):.4f}')
print(f'F1 churn : {f1_score(y_test, y_pred, pos_label=1):.4f}')
print(f'Recall   : {recall_score(y_test, y_pred, pos_label=1):.4f}')
print(f'Precision: {precision_score(y_test, y_pred, pos_label=1, zero_division=0):.4f}')
print()
print(classification_report(y_test, y_pred, target_names=['Not Churned', 'Churned']))

## 3. SHAP Explainability

SHAP (SHapley Additive exPlanations) explains **why** the model made each prediction. Plots:
- **Summary (beeswarm)** — global feature importance + direction of effect
- **Bar plot** — mean absolute SHAP value per feature
- **Waterfall** — single prediction breakdown

In [ ]:
print('Computing SHAP values on 5,000 test samples...')
shap_sample_idx = np.random.choice(len(X_test), size=5000, replace=False)
X_test_sample = X_test[shap_sample_idx]

explainer   = shap.TreeExplainer(tuned_model)
shap_values = explainer(X_test_sample)

print('SHAP values computed.')

In [ ]:
# Global summary — beeswarm
fig, ax = plt.subplots(figsize=(10, 8))
shap.summary_plot(shap_values.values, X_test_sample,
                  feature_names=PROC_COLS, show=False)
plt.title('SHAP Summary — Global Feature Impact on Churn Prediction',
          fontweight='bold', pad=12)
plt.tight_layout(); plt.show()

In [ ]:
# Bar plot — mean |SHAP|
fig, ax = plt.subplots(figsize=(9, 7))
shap.summary_plot(shap_values.values, X_test_sample,
                  feature_names=PROC_COLS, plot_type='bar', show=False)
plt.title('Mean |SHAP| — Feature Importance (Global)', fontweight='bold', pad=12)
plt.tight_layout(); plt.show()

In [ ]:
# Waterfall for the highest-risk churner in the sample
pred_probs_sample = tuned_model.predict_proba(X_test_sample)[:, 1]
highest_risk_idx  = np.argmax(pred_probs_sample)

print(f'Predicted churn probability: {pred_probs_sample[highest_risk_idx]:.3%}')
print(f'Actual label: {y_test[shap_sample_idx[highest_risk_idx]]}')

shap.waterfall_plot(shap_values[highest_risk_idx], max_display=12, show=True)

In [ ]:
# Dependence plots for the top 2 SHAP features
mean_abs_shap = np.abs(shap_values.values).mean(axis=0)
top2_idx = np.argsort(mean_abs_shap)[::-1][:2]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, idx in zip(axes, top2_idx):
    feat_name = PROC_COLS[idx]
    shap.dependence_plot(idx, shap_values.values, X_test_sample,
                         feature_names=PROC_COLS, ax=ax, show=False)
    ax.set_title(f'SHAP Dependence: {feat_name}', fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
os.makedirs('artifacts', exist_ok=True)
joblib.dump(tuned_model, 'artifacts/model_tuned_final.pkl')

# Save best params and threshold
with open('artifacts/best_params.json', 'w') as f:
    json.dump({'model': best_model_name,
               'params': study.best_params,
               'threshold': float(best_t),
               'val_roc_auc': study.best_value}, f, indent=2)

print('Saved:')
print('  artifacts/model_tuned_final.pkl')
print('  artifacts/best_params.json')